<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 18 · Technology Stack and Deployment Patterns

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Notebook Goals

This notebook turns Chapter 18 into a small but realistic daily pipeline prototype: config, logging, feature engineering, scoring, and alerting.

### Getting Help While Thinking About Deployment

- Chapter 3 reviews environments and project layout.
- Appendix F lists additional tooling patterns you may want to adopt.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"

import datetime as dt
import logging

### Data Loading
For completeness we keep the price panel available for quick tests.

In [ ]:
prices = pd.read_csv(DATA_PATH,
parse_dates=["Date"]).set_index("Date").sort_index().ffill()

## 1. Configuration Helper

A central config dictionary keeps file paths and simple settings in one place so that scripts and notebooks stay in sync.

In [ ]:
CONFIG = {
    "data_file": DATA_PATH,
    "features_file": Path("../data/features/pyaiam_features.parquet"),
    "reports_dir": Path("../reports"),
    "recipients": ["research@tpq.io"],
}
CONFIG

### 1.1 Logger Setup

Structured logging makes it easy to debug and audit pipeline runs.

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)
logger = logging.getLogger("pipeline")
logger.info("Logger initialized")

## 2. Daily Pipeline Skeleton

We tie together ingestion, feature engineering, and scoring, mirroring a simple daily research run.

In [ ]:
def ingest(path: Path) -> pd.DataFrame:
    logger.info("Loading %s", path)
    return pd.read_csv(path, parse_dates=["Date"]).set_index("Date").sort_index()

def engineer(df: pd.DataFrame) -> pd.DataFrame:
    logger.info("Engineering rolling features")
    log_ret = np.log(df / df.shift(1))
    vol = log_ret.rolling(20).std()
    momentum = df.pct_change(20, fill_method=None)
    return pd.concat({"vol": vol, "mom": momentum}, axis=1).dropna()

def score(panel: pd.DataFrame) -> pd.DataFrame:
    logger.info("Scoring placeholder model")
    return panel.groupby(level=0).mean()

def daily_pipeline() -> None:
    raw = ingest(CONFIG["data_file"])
    feats = engineer(raw)
    scores = score(feats)
    out = CONFIG["reports_dir"] / f"scores_{dt.date.today():%Y%m%d}.parquet"
    out.parent.mkdir(parents=True, exist_ok=True)
    scores.to_parquet(out)
    logger.info("Saved scores to %s", out)

daily_pipeline()

## 3. Alert Stub

A simple alert function demonstrates how you might route messages to email, chat, or monitoring systems.

In [ ]:
def send_alert(message: str) -> None:
    logger.warning("ALERT: %s", message)

send_alert("Pipeline completed (demo)")

## 4. Scheduling Snippet

The following crontab line shows how you could run the pipeline every weekday at 06:00 UTC.

In [ ]:
import textwrap
crontab = textwrap.dedent("""
# Run pipeline weekdays at 06:00 UTC
0 6 * * 1-5 /usr/bin/python /repo/scripts/run_pipeline.py >> /repo/logs/pipeline.log
2>&1
""")
print(crontab)

## 5. Exercises

### Exercise 1 – Dependency Guardrail
Check that key package versions match expectations before running the pipeline.
<details><summary>Hint</summary>Use importlib.metadata.version and compare against a small dict of expected versions.</details>

### Exercise 2 – Data Freshness Check
Warn if the latest data point is older than a configured threshold.
<details><summary>Hint</summary>Compare df.index.max().date() to dt.date.today().</details>

### Exercise 3 – CLI Entry Point
Write a main function that parses a <code>--dry-run</code> flag and routes to daily_pipeline accordingly.
<details><summary>Hint</summary>Use argparse and wire the result into a small if/else block.</details>

## 6. Takeaways

Small, composable helpers for configuration, logging, and scheduling make it much easier to move from research notebooks to production pipelines.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">